# Pelajaran 11 - Protokol Ejen-ke-Ejen (A2A)


## Persediaan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Apakah Protokol A2A?

**Protokol Ejen-ke-Ejen (A2A)** adalah piawaian terbuka yang membolehkan ejen AI berkomunikasi,
mengenal pasti satu sama lain, dan bekerjasama — walaupun mereka dibina menggunakan rangka kerja berbeza
atau dihoskan oleh perkhidmatan yang berbeza.

Konsep utama:

- **Penemuan** – Ejen menerbitkan *Kad Ejen* yang menerangkan keupayaan mereka, memudahkan
  ejen lain (atau pengatur cara) untuk mencari pakar yang tepat untuk sesuatu tugasan.
- **Pemindahan Mesej** – Ejen bertukar mesej berstruktur melalui protokol yang sama, jadi
  permintaan dari satu ejen boleh difahami dan dipenuhi oleh ejen lain tanpa mengira
  pelaksanaan dalaman ejen tersebut.
- **Kitar Hayat Tugasan** – A2A mentakrifkan keadaan seperti *dihantar*, *sedang dijalankan*, *lengkap*, dan
  *gagal*, memberikan pengatur cara visibiliti penuh tentang bagaimana tugasan yang diwakilkan sedang berkembang.

Dalam pelajaran ini, kita mensimulasikan kerjasama gaya A2A dengan menghubungkan tiga ejen pelancongan pakar
ke dalam aliran kerja di mana setiap ejen menyumbang kepakarannya dan menyerahkan keputusan kepada yang seterusnya.


## Mewujudkan Ejen Pelancongan Khusus


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Kolaborasi Multi-Ejen melalui Aliran Kerja

Kami menyambungkan tiga ejen ke dalam aliran kerja berurutan yang mencerminkan penghantaran mesej A2A:

1. **CurrencyExchangeAgent** menerima permintaan pengguna dan menghasilkan panduan mata wang.
2. **ActivityPlannerAgent** menerima konteks yang diperkaya dan menambah cadangan aktiviti.
3. **TravelManagerAgent** mensintesis kedua-dua input menjadi taklimat perjalanan akhir.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Memahami A2A dalam Produksi

Dalam persekitaran produksi, protokol A2A membuka senario silang-perkhidmatan yang berkuasa:

| Keupayaan | Perihal |
|---|---|
| **Interop silang-rangka kerja** | Ejen yang dibina dengan satu rangka kerja boleh mendelegasikan tugas kepada ejen yang dibina dengan sebarang rangka kerja A2A-kompatibel yang lain, membolehkan interoperabiliti sebenar merentas organisasi. |
| **Sempadan perkhidmatan** | Ejen boleh wujud dalam mikroservis berasingan, rantau awan, atau bahkan organisasi berlainan sambil terus berkolaborasi dengan lancar. |
| **Penemuan dinamik** | Pengurus tugasan boleh membuat pertanyaan ke pendaftar Kad Ejen semasa runtime untuk mencari pakar yang paling sesuai bagi sub-tugas tertentu. |
| **Penstriman & pemberitahuan tolak** | A2A menyokong Server-Sent Events (SSE) untuk kemas kini kemajuan masa nyata dan pemberitahuan tolak untuk tugas yang berjalan lama. |

Aliran kerja yang kami bina di atas adalah versi mudah, dalam-proses bagi corak ini. Dalam
pelaksanaan sebenar, setiap ejen akan mendedahkan titik hujung HTTP, menerbitkan Kad Ejen, dan berkomunikasi
melalui protokol A2A JSON-RPC.


## Ringkasan

Dalam pelajaran ini anda telah belajar:

1. **Apa itu protokol A2A** — satu standard terbuka untuk penemuan, pemesejan,
   dan pengurusan tugas antara ejen.
2. **Cara membuat ejen khusus** — ejen Penukaran Mata Wang, ejen Perancang Aktiviti,
   dan pengendali Pengurus Perjalanan.
3. **Cara menyambungkan ejen dalam aliran kerja** — menggunakan `WorkflowBuilder` untuk memodelkan penghantaran mesej secara berurutan antara ejen.

4. **Bagaimana A2A berfungsi dalam pengeluaran** — membolehkan kerjasama merentasi rangka kerja, merentasi perkhidmatan
   dengan penemuan dinamik dan kemas kini aliran.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
